# Quasi-norm Muon: run Hypothesis 1 on Colab

**Runtime → Change runtime type → GPU** (A100 if you can get it; T4 works but is much slower).
Run top to bottom. Every sweep is **resumable**: if Colab disconnects, re-run the setup cells and the
sweep cell again and it skips finished runs (results are saved to Drive if you mount it).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 1. Setup (results go to Drive if mounted)

In [ ]:
import os
USE_DRIVE = True   # set False to keep results only on the (ephemeral) Colab disk
RESULTS = "/content/results"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/cs182_results"
os.makedirs(RESULTS, exist_ok=True)
print("results dir:", RESULTS)

In [ ]:
%cd /content
!git clone https://github.com/BistroP/CS182-final-project.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!python -m pytest -q tests

In [ ]:
# Downloads tinyshakespeare (~1.1 MB) from github.com/karpathy/char-rnn
!python prepare_shakespeare.py

## 2. Benchmark (≈ 3–6 min) — do this first
Look at the **freon svd vs eigh** rows (`opt_step` and `s/iter`) and pick the faster one for `FREON_IMPL` below.
Please paste the output back to the team: it decides how big a sweep we can afford.

In [ ]:
!python bench.py --iters 100 --out {RESULTS}/bench.json

In [ ]:
FREON_IMPL = "eigh"   # <- set to "svd" if the benchmark shows svd is faster

## 3. Stage 1 — tune learning rates (seed 42, 1500 iters per run)
16 `(matrix_lr, base_lr)` points per spectral optimizer (15 for SGD / TruncatedSGD, which have their own larger-lr grid).
That's 16×4 + 15×2 = 94 short runs. Kaon is Muon-cost; Freon cost depends on the benchmark.

In [ ]:
OPTS = "muon kaon freon:0.6667 freon:0.75 sgd truncsgd:0.05"
!python sweep.py tune --opts {OPTS} --max_iters 1500 --freon_impl {FREON_IMPL} --out {RESULTS}/tune.jsonl

In [ ]:
!python sweep.py best --out {RESULTS}/tune.jsonl

## 4. Stage 2 — final runs, 3 seeds × 5000 iters at each optimizer's best lrs
Paste the `spec:matrix_lr:base_lr` lines printed by the cell above into `FINAL_SPECS`.
(`--rhos 1` = no rescaling. For the paper's Figure-2 learning-rate sensitivity, drop `--rhos 1`.)

In [ ]:
FINAL_SPECS = "muon:0.02:0.002 kaon:0.02:0.002 freon:0.6667:0.02:0.002 freon:0.75:0.02:0.002 sgd:0.2:0.002 truncsgd:0.05:0.2:0.002"  # <- REPLACE with the best specs from stage 1
!python sweep.py scale --opts {FINAL_SPECS} --rhos 1 --seeds 42 43 44 --freon_impl {FREON_IMPL} --out {RESULTS}/final.jsonl

## 5. Summary: mean ± std final validation loss over seeds

In [ ]:
import json, collections, statistics as st
rows = [json.loads(l) for l in open(f"{RESULTS}/final.jsonl")]
by = collections.defaultdict(list)
for r in rows:
    by[r["tag"]].append(r["final_val_loss"])
for tag, v in sorted(by.items(), key=lambda kv: st.mean(kv[1])):
    sd = st.stdev(v) if len(v) > 1 else float("nan")
    print(f"{tag:16s} {st.mean(v):.4f} ± {sd:.4f}   (n={len(v)})")